# Verifier fine-tuning and evaluation (Kaggle, T4 x2)

One session does the whole verifier step (`docs/VERIFIER_PLAN.md`):
1. generates the training data from the training-only assignments TR1 and TR2
   (`verifier.train_data`) and the evaluation sets: the dev claims with the evidence the
   pipeline gathered in `results/pipeline_dev_v2/` and `v3/`, and the held-out authority tone;
2. trains Qwen2.5-3B-Instruct with QLoRA on GPU 0 and merges the adapter (`verifier.train`);
3. installs vLLM and evaluates the untuned 3B (A4a), the fine-tuned 3B and the prompted
   7B (A4b) on the same claims (`verifier.evaluate`);
4. runs the full pipeline with the fine-tuned verifier on dev (`configs/pipeline_ft_dev.yaml`).

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*. About 2 to 2.5 hours.
Outputs: `/kaggle/working/verifier_ft_outputs.zip` (metrics, events, predictions, logs:
download this) and `/kaggle/working/verifier_model/` (the merged model, about 6 GB, kept in
the notebook output for the test runs, never committed).

In [ ]:
REPO_URL = "https://github.com/JagadishReddyK7/regrade-appeals-agent.git"
BRANCH = "feat/verifier-training"
VLLM_SPEC = "vllm==0.32.0"  # as in notebooks/pilot_dev.ipynb
WORKDIR = "/kaggle/temp/regrade-appeals-agent"
OUT = "/kaggle/working/verifier_ft_outputs"
MODEL_DIR = "/kaggle/working/verifier_model"  # merged fine-tuned verifier
SERVED_PATH = "/kaggle/temp/verifier-ft"  # configs/pipeline_ft_dev.yaml serves this path
RUN_PIPELINE = True  # step 4: the full system with the fine-tuned verifier on dev

In [ ]:
# Clone. The token (if any) is removed from the git remote afterwards.
import glob
import json
import os
import shutil
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, or no secret configured
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(WORKDIR)
os.makedirs(OUT, exist_ok=True)
# Number the GPUs the same way in CUDA and nvidia-smi (servers pick a GPU by index).
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
!git log -1 --oneline

In [ ]:
!pip install -q -e ".[gpu]"
# Kaggle's torchao 0.10 makes peft refuse to merge the adapter into the float16 model
# (peft needs torchao >= 0.16 if it is installed); nothing here uses torchao.
!pip uninstall -y -q torchao
!python -c "import sys, torch; print(sys.version, 'torch', torch.__version__, torch.version.cuda)"
!pip list 2>/dev/null | grep -iE "^(transformers|peft|bitsandbytes|accelerate) "
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data

In [ ]:
def run(*args):
    done = subprocess.run(["python", "-m", *args], capture_output=True, text=True)
    print(done.stdout[-6000:], done.stderr[-6000:])
    return done.returncode


assert run("verifier.train_data", "--out", "runs/verifier_train.jsonl",
           "--heldout", "runs/verifier_heldout_authority.jsonl") == 0
EVAL_SETS = {}
for version in ("v2", "v3"):
    run_dir = glob.glob(f"results/pipeline_dev_{version}/pipeline_dev-*")[0]
    EVAL_SETS[f"dev_{version}"] = f"runs/eval_dev_{version}.jsonl"
    assert run("verifier.eval_data", "--run", run_dir, "--cases", "data/cases/dev",
               "--out", EVAL_SETS[f"dev_{version}"]) == 0
EVAL_SETS["heldout"] = "runs/eval_heldout_authority.jsonl"
assert run("verifier.eval_data", "--examples", "runs/verifier_heldout_authority.jsonl",
           "--out", EVAL_SETS["heldout"]) == 0
print(EVAL_SETS)

## 2. Train (GPU 0) and merge
About 40 minutes (300 steps of 2 examples, about 1,400 tokens each); the output appears
only when training ends. `metrics.json` shows the loss and a generation check on 6
examples; the adapter is copied to the saved output at once.

**Reusing a trained adapter:** if an earlier version of this notebook trained the adapter,
add that version's output as an input (*Add Input* -> *Your Work*); the cell finds
`runs/verifier_train-*/adapter` there, skips training and only merges.

In [ ]:
TRAIN_CONFIG = "configs/verifier_train.yaml"
found = sorted(glob.glob("/kaggle/input/**/runs/verifier_train-*/adapter/adapter_config.json",
                         recursive=True))
if found:  # reuse the adapter of an earlier version: copy it (inputs are read-only), merge
    source = os.path.dirname(os.path.dirname(found[-1]))
    target = f"runs/{os.path.basename(source)}"
    shutil.copytree(source, target, dirs_exist_ok=True)
    print("reusing the trained adapter from", source)
    train_code = run("verifier.train", "--config", TRAIN_CONFIG, "--merge-only", target)
else:
    train_code = run("verifier.train", "--config", TRAIN_CONFIG)
train_dirs = sorted(glob.glob("runs/verifier_train-*"))
# Keep the training run (adapter included, merged model excluded) in the saved output
# at once, so that a later failure cannot lose it.
for train_dir in train_dirs:
    shutil.copytree(train_dir, f"{OUT}/runs/{os.path.basename(train_dir)}", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("merged"))
assert train_code == 0 and train_dirs, "training or merge failed: see the output above"
train_dir = train_dirs[-1]
print(open(f"{train_dir}/metrics.json").read())
assert os.path.isdir(f"{train_dir}/merged"), "merge failed: see the output above"
if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
shutil.move(f"{train_dir}/merged", MODEL_DIR)
if os.path.lexists(SERVED_PATH):
    os.remove(SERVED_PATH)
os.symlink(MODEL_DIR, SERVED_PATH)
print(os.listdir(MODEL_DIR))

## 3. Evaluate the verifiers on the same claims
Installs vLLM, which brings its own PyTorch (training is finished, so this is safe).

In [ ]:
!pip install -q "{VLLM_SPEC}"
!pip uninstall -y -q torchaudio
!python -c "import torch, vllm; print('gpu', torch.cuda.is_available(), 'vllm', vllm.__version__)"

In [ ]:
import signal
import time


def gpu_used_mb(gpu):
    query = ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits",
             "-i", str(gpu)]
    return int(subprocess.run(query, capture_output=True, text=True).stdout.strip() or 0)


def stop_server(stem, gpu):
    # Stop a vLLM server and its worker processes (one process group) and wait until the
    # GPU is free. Returns False if the GPU is still busy after two minutes.
    if os.path.exists(f"{stem}.pid"):
        try:
            os.killpg(int(open(f"{stem}.pid").read()), signal.SIGTERM)
        except ProcessLookupError:
            pass
        os.remove(f"{stem}.pid")
    for _ in range(60):
        if gpu_used_mb(gpu) < 1000:
            return True
        time.sleep(2)
    print(f"GPU {gpu} is still busy after stopping {stem}:")
    subprocess.run(["nvidia-smi"])
    return False


def keep_log(stem, label):
    if os.path.exists(f"{stem}.log"):
        shutil.copy(f"{stem}.log", f"{OUT}/{label}.log")


def evaluate_all(name, base_url, model):
    for set_name, path in EVAL_SETS.items():
        run("verifier.evaluate", "--examples", path, "--base-url", base_url,
            "--model", model, "--name", f"{name}_{set_name}")


# A failing server is reported and skipped (no exception), so the outputs are always saved.
# Untuned 3B (A4a) on GPU 1.
if run("llm.serve", "--config", "configs/verifier_3b_base.yaml", "--verifier") == 0:
    evaluate_all("base3b", "http://localhost:8001/v1", "Qwen/Qwen2.5-3B-Instruct")
else:
    print("UNTUNED 3B DID NOT START: see the log above")
keep_log("vllm_verifier", "vllm_base3b")
stop_server("vllm_verifier", gpu=1)
# Fine-tuned 3B on GPU 1 (stays up for the pipeline run).
FT_READY = run("llm.serve", "--config", "configs/pipeline_ft_dev.yaml", "--verifier") == 0
if FT_READY:
    evaluate_all("ft3b", "http://localhost:8001/v1", SERVED_PATH)
else:
    print("FINE-TUNED 3B DID NOT START: see the log above")
# Prompted 7B (A4b) on GPU 0: the same server the pipeline uses.
MAIN_READY = run("llm.serve", "--config", "configs/pipeline_ft_dev.yaml") == 0
if MAIN_READY:
    evaluate_all("prompted7b", "http://localhost:8000/v1", "Qwen/Qwen2.5-7B-Instruct-AWQ")
else:
    print("7B DID NOT START: see the log above")

In [ ]:
# Verifier results per verifier and evaluation set.
for path in sorted(glob.glob("runs/verifier_eval-*/metrics.json")):
    m = json.load(open(path))
    print(f"{m['run_id'][:45]:45s} n={m['n']:3d} acc={m['accuracy']:.2f} "
          f"macroF1={m['macro_f1']:.2f} invalid={m['invalid']} tone_sens={m['tone_sensitivity']}")

## 4. Full pipeline with the fine-tuned verifier (dev)

In [ ]:
if RUN_PIPELINE and FT_READY and MAIN_READY:
    if run("llm.smoke", "--config", "configs/pipeline_ft_dev.yaml") == 0:
        run("eval.run", "--config", "configs/pipeline_ft_dev.yaml")
    else:
        print("SMOKE TEST FAILED: pipeline run skipped")
    for path in sorted(glob.glob("runs/pipeline_ft_dev-*/metrics.json")):
        o = json.load(open(path))["overall"]
        keys = ("exact_decision_accuracy", "wrongful_change_rate", "wrongful_refusal_rate",
                "escalation_recall", "pushback_flip_rate")
        print({k: o.get(k) for k in keys})

## Save outputs and stop the servers
The adapter and the merged model are not in the zip; the merged model stays in `verifier_model/`.

In [ ]:
os.makedirs(f"{OUT}/runs", exist_ok=True)
for path in glob.glob("runs/*"):
    target = f"{OUT}/runs/{os.path.basename(path)}"
    if os.path.isdir(path):
        shutil.copytree(path, target, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("adapter", "merged"))
    elif path.endswith(".jsonl"):
        shutil.copy(path, target)
keep_log("vllm", "vllm_7b")
keep_log("vllm_verifier", "vllm_ft3b")
!pip freeze > {OUT}/pip_freeze.txt
!git rev-parse HEAD > {OUT}/commit.txt
shutil.make_archive("/kaggle/working/verifier_ft_outputs", "zip", OUT)
stop_server("vllm", gpu=0)
stop_server("vllm_verifier", gpu=1)
print("done")